<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 컴퓨터 비전 · 00. LeNet-5

## Gradient-Based Learning Applied to Document Recognition

- **원 논문:** [Gradient-Based Learning Applied to Document Recognition](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf)
- **저자 / 발표:** Yann LeCun, Léon Bottou, Yoshua Bengio, Patrick Haffner · Proceedings of the IEEE 86(11) (1998)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 공유 커널과 평균 subsampling으로 C1-S2-C3-S4 경로를 재현한다.

**축소하거나 생략한 부분:** 원 논문의 32×32 문자, 부분 연결 C3, 학습 가능한 subsampling, RBF 출력 대신 16×16 합성 도형과 완전 연결 convolution, 4-class linear head를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §II.A: local receptive fields와 shared weights | Task 1 Conv2d | 커널/shape assert |
| §II.B, Fig. 2: C1-S2-C3-S4-C5-F6 | Task 2 LeNetMini | 중간 shape assert |
| §II.B: average subsampling | Task 2 AvgPool2d | max pool과 구별 |
| Table I: gradient-based training | Task 3 | loss/accuracy plot |

## 핵심 아이디어와 수식

$$h_{c,i,j}=\tanh\!\left(b_c+\sum_{k,u,v}W_{c,k,u,v}x_{k,i+u,j+v}\right)$$

**기호 해설:** x는 입력, W는 공유 커널, b는 편향, h는 특징 맵이다.

**코드 대응:** Task 1은 공간 크기를 계산하고 Task 2의 LeNetMini가 C1-S2-C3-S4를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → C1/S2 [B,6,6,6] → C3/S4 [B,16,1,1] → [B,4].

**포트폴리오 구현 범위:** 원 논문의 부분 연결과 RBF head는 생략하지만 평균 pooling과 공유 가중치는 보존한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §II.A: local receptive fields와 shared weights
- **노트북에서 구현할 부분:** Task 1 Conv2d
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 커널/shape assert

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 1: convolution/pooling 출력 크기 함수와 설정 dataclass를 구현하세요.
@dataclass
class LeNetConfig:
    pass


def spatial_size(size, kernel, stride=1, padding=0):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §II.B, Fig. 2: C1-S2-C3-S4-C5-F6
- **노트북에서 구현할 부분:** Task 2 LeNetMini
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 중간 shape assert

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 2: C1-S2-C3-S4와 classifier를 갖는 LeNetMini를 구현하세요.
class LeNetMini(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward_features(self, inputs):
        """C1-S2-C3-S4 특징 맵 [B, C, 1, 1]을 반환한다."""
        raise NotImplementedError

    def forward(self, inputs):
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** Table I: gradient-based training
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** loss/accuracy plot

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 3: 분류 손실, 한 학습 step, 평가 함수를 각각 구현하세요.
def classification_loss(logits, targets):
    raise NotImplementedError


def train_step(model, optimizer, inputs, targets):
    raise NotImplementedError


def evaluate_classifier(model, inputs, targets):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.